# Homework 1

Since I don't have a GPU, I've collected all the code here so I can compile and run the CUDA examples on Google Colab.

In [1]:
%%shell

nvidia-smi

## 1. Hello World

Fill the corresponding thread number and block number so that the expected result prints:

```text
Hello from block: 0, thread: 0
Hello from block: 0, thread: 1
Hello from block: 1, thread: 0
Hello from block: 1, thread: 1
```

In [1]:
%%writefile hello.cu

#include <stdio.h>

__global__ void hello() {
  printf("Hello from block: %u, thread: %u\n", blockIdx.x, threadIdx.x);
}

int main(){

  hello<<<2, 2>>>();
  cudaDeviceSynchronize();
}

Writing hello.cu


In [2]:
%%shell

nvcc hello.cu -o hello
./hello

Hello from block: 0, thread: 0
Hello from block: 0, thread: 1
Hello from block: 1, thread: 0
Hello from block: 1, thread: 1


## 2. Vector Add

Write a vector add program such that `C = A + B`.

In [3]:
%%writefile vector_add.cu

#include <stdio.h>

__global__ void vector_add(const float *a, const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}

const int DSIZE = 4096;
const int block_size = 256;

int main() {
    float *h_A, *h_B, *h_C; // host vectors
    float *d_A, *d_B, *d_C; // device vectors

    // Allocate host memory
    h_A = new float[DSIZE];
    h_B = new float[DSIZE];
    h_C = new float[DSIZE];

    for (int i = 0; i < DSIZE; ++i) {
        h_A[i] = rand()/(float)RAND_MAX;
        h_B[i] = rand()/(float)RAND_MAX;
        h_C[i] = 0.0f;
    }

    // Allocate device memory
    cudaMalloc(&d_A, DSIZE * sizeof(float));
    cudaMalloc(&d_B, DSIZE * sizeof(float));
    cudaMalloc(&d_C, DSIZE * sizeof(float));

    cudaMemcpy(d_A, h_A, DSIZE * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, DSIZE * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_C, h_C, DSIZE * sizeof(float), cudaMemcpyHostToDevice);

    vector_add<<<(DSIZE+block_size-1)/block_size, block_size>>>(d_A, d_B, d_C, DSIZE);

    cudaDeviceSynchronize();
    cudaMemcpy(h_C, d_C, DSIZE * sizeof(float), cudaMemcpyDeviceToHost);

    printf("First 10 results:\n");
    for (int i = 0; i < 10; ++i) {
        printf("%f + %f = %f (should be %f)\n", h_A[i], h_B[i], h_C[i], h_A[i] + h_B[i]);
    }
    
    // Free device memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    // Free host memory
    delete[] h_A;
    delete[] h_B;
    delete[] h_C;

    return 0;
}

Writing vector_add.cu


In [4]:
%%shell
nvcc vector_add.cu -o vector_add
./vector_add

First 10 results:
0.840188 + 0.394383 = 1.234571 (should be 1.234571)
0.783099 + 0.798440 = 1.581539 (should be 1.581539)
0.911647 + 0.197551 = 1.109199 (should be 1.109199)
0.335223 + 0.768230 = 1.103452 (should be 1.103452)
0.277775 + 0.553970 = 0.831745 (should be 0.831745)
0.477397 + 0.628871 = 1.106268 (should be 1.106268)
0.364784 + 0.513401 = 0.878185 (should be 0.878185)
0.952230 + 0.916195 = 1.868425 (should be 1.868425)
0.635712 + 0.717297 = 1.353009 (should be 1.353009)
0.141603 + 0.606969 = 0.748571 (should be 0.748571)


## 3. Matrix Multiply (naive)
Write a naive matrix multiply program such that `C = A * B`.

In [7]:
%%writefile matrix_mul.cu

#include <stdio.h>

#define cudaCheckErrors(msg) \
    do { \
        cudaError_t __err = cudaGetLastError(); \
        if (__err != cudaSuccess) { \
            fprintf(stderr, "Fatal error: %s (%s at %s:%d)\n", \
                msg, cudaGetErrorString(__err), \
                __FILE__, __LINE__); \
            fprintf(stderr, "*** FAILED - ABORTING\n"); \
            exit(1); \
        } \
    } while (0)

const int DSIZE = 8192;
const int block_size = 32;
const float A_val = 1.0f;
const float B_val = 2.0f;

// matrix multiply: A.row dot product B.col
__global__ void matrix_mul(float *A, float *B, float *C, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    
    if (row < n && col < n) {
        float sum = 0.0f;
        for (int k = 0; k < n; ++k) {
            sum += A[row * n + k] * B[k * n + col];
        }
        C[row * n + col] = sum;
    }
}

int main() {
    float *h_A, *h_B, *h_C; // host matrices
    float *d_A, *d_B, *d_C; // device matrices

    h_A = new float[DSIZE * DSIZE];
    h_B = new float[DSIZE * DSIZE];
    h_C = new float[DSIZE * DSIZE];

    for (int i = 0; i < DSIZE * DSIZE; ++i) {
        h_A[i] = A_val;
        h_B[i] = B_val;
        h_C[i] = 0.0f;
    }

    cudaMalloc(&d_A, DSIZE * DSIZE * sizeof(float));
    cudaMalloc(&d_B, DSIZE * DSIZE * sizeof(float));
    cudaMalloc(&d_C, DSIZE * DSIZE * sizeof(float));
    cudaMemcpy(d_A, h_A, DSIZE * DSIZE * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, DSIZE * DSIZE * sizeof(float), cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(block_size, block_size);
    dim3 numBlocks((DSIZE + threadsPerBlock.x - 1) / threadsPerBlock.x, (DSIZE + threadsPerBlock.y - 1) / threadsPerBlock.y);
    matrix_mul<<<numBlocks, threadsPerBlock>>>(d_A, d_B, d_C, DSIZE);

    cudaDeviceSynchronize();
    cudaMemcpy(h_C, d_C, DSIZE * DSIZE * sizeof(float), cudaMemcpyDeviceToHost);

    cudaCheckErrors("kernel execution failure or cudaMemcpy H2D failure");
    for (int i = 0; i < DSIZE*DSIZE; i++) if (h_C[i] != A_val*B_val*DSIZE) {printf("mismatch at index %d, was: %f, should be: %f\n", i, h_C[i], A_val*B_val*DSIZE); return -1;}
    printf("Success!\n"); 

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    delete[] h_A;
    delete[] h_B;
    delete[] h_C;
}

Overwriting matrix_mul.cu


In [8]:
%%shell

nvcc matrix_mul.cu -o matrix_mul
./matrix_mul

Success!
